# Geração de Embeddings e Clusters de Genes

Notebook orquestrador que chama `gene_clustering.py` para gerar embeddings OWL2Vec* e clusters (KMeans + HDBSCAN) para cada dataset do projeto.

**Datasets disponíveis:** BLCA, BRCA, LIHC, PRAD

**Saídas geradas para cada dataset:**
| Arquivo | Local |
|---|---|
| `{dataset}_embeddings_mowl.csv` | `data/output/embeddings/` |
| `{dataset}_umap_mowl.csv` | `data/output/umap/` |
| `genes_clustered_{dataset}.csv` | `data/output/clusters/` |
| `clusters_{dataset}.png` | `data/output/clusters/` |

---
> **Requisitos:** Java JDK instalado + `pip install mowl-borg mygene gensim scikit-learn umap-learn hdbscan matplotlib`

## 1. Inicialização

> ⚠️ A JVM é inicializada dentro de `gene_clustering.py` no momento do import. Esse import **deve ser feito apenas uma vez** por sessão do kernel.

In [ ]:
# Importa o pipeline — a JVM é iniciada aqui (mowl.init_jvm)
from gene_clustering import (
    gera_clusters_genes_com_mowl,
    OUT_EMBEDDINGS,
    OUT_CLUSTERS,
    OUT_UMAP,
)

from pathlib import Path
import pandas as pd

## 2. Configuração dos Datasets

In [ ]:
# Datasets a processar — remova ou comente os que não deseja rodar
DATASETS = [
    "BLCA",
    "BRCA",
    "LIHC",
    "PRAD",
]

print(f"Datasets configurados : {DATASETS}")
print(f"Embeddings  → {OUT_EMBEDDINGS.resolve()}")
print(f"Clusters    → {OUT_CLUSTERS.resolve()}")
print(f"UMAP        → {OUT_UMAP.resolve()}")

## 3. Execução do Pipeline

Para cada dataset, o pipeline executa:
1. Download do `go.owl` → `data/ontology/` (apenas na primeira vez)
2. Busca de anotações GO via mygene.info
3. Projeção OWL → grafo + random walks → `data/cache/`
4. Treinamento Word2Vec (OWL2Vec*)
5. Embeddings por gene → `data/output/embeddings/`
6. Clusterização KMeans + HDBSCAN
7. Visualização UMAP + exportação → `data/output/clusters/` e `data/output/umap/`

In [ ]:
failed = []

for dataset in DATASETS:
    print(f"\n{'='*60}")
    print(f"  Processando dataset: {dataset}")
    print(f"{'='*60}")
    try:
        gera_clusters_genes_com_mowl(dataset)
        print(f"✅ {dataset} concluído com sucesso.")
    except Exception as e:
        print(f"❌ Erro em {dataset}: {e}")
        failed.append((dataset, str(e)))

print(f"\n{'='*60}")
if failed:
    print(f"Pipeline finalizado com {len(failed)} erro(s):")
    for ds, err in failed:
        print(f"  - {ds}: {err}")
else:
    print(f"Pipeline finalizado! Todos os {len(DATASETS)} datasets processados com sucesso.")

## 4. Resumo dos Resultados

Carrega e exibe um resumo dos clusters gerados para cada dataset.

In [ ]:
summary_rows = []

for dataset in DATASETS:
    csv_path = OUT_CLUSTERS / f"genes_clustered_{dataset}.csv"
    if not csv_path.exists():
        print(f"⚠️  {dataset}: arquivo não encontrado ({csv_path})")
        continue

    df = pd.read_csv(csv_path)
    n_genes       = len(df)
    n_kmeans      = df["cluster_kmeans"].nunique()
    n_hdbscan     = df["cluster_hdbscan"].nunique() - (1 if -1 in df["cluster_hdbscan"].values else 0)
    noise_pct     = (df["cluster_hdbscan"] == -1).mean() * 100

    summary_rows.append({
        "Dataset":             dataset,
        "Genes com embedding": n_genes,
        "Clusters KMeans":     n_kmeans,
        "Clusters HDBSCAN":    n_hdbscan,
        "Ruído HDBSCAN (%)": f"{noise_pct:.1f}",
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df.set_index("Dataset"))

## 5. Visualizações Geradas

Exibe os gráficos UMAP salvos para cada dataset.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

available = [
    (ds, OUT_CLUSTERS / f"clusters_{ds}.png")
    for ds in DATASETS
    if (OUT_CLUSTERS / f"clusters_{ds}.png").exists()
]

if not available:
    print("Nenhuma visualização encontrada. Execute a célula de pipeline primeiro.")
else:
    fig, axes = plt.subplots(len(available), 1, figsize=(16, 7 * len(available)))
    if len(available) == 1:
        axes = [axes]

    for ax, (ds, img_path) in zip(axes, available):
        img = mpimg.imread(img_path)
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(ds, fontsize=14, fontweight="bold", pad=8)

    plt.tight_layout()
    plt.show()